# W08 — Assignment (SQL limpieza + Many-to-Many)

**Único entregable semanal.**

## Setup

In [1]:
from pathlib import Path
import duckdb
import os

PROJECT_ROOT = Path(r"C:\Users\USUARIO WINDOWS\CODIGOS\PAZ CD").resolve()
os.chdir(PROJECT_ROOT)

RAW_CSV = PROJECT_ROOT / "data" / "raw" / "pscomppars.csv"
DB_PATH = PROJECT_ROOT / "data" / "exoplanets_w08.duckdb"
DOCS_DIR = PROJECT_ROOT / "docs"
ART_DIR = PROJECT_ROOT / "artifacts"

DOCS_DIR.mkdir(parents=True, exist_ok=True)
ART_DIR.mkdir(parents=True, exist_ok=True)

if not RAW_CSV.exists():
    raise FileNotFoundError(f"Missing {RAW_CSV}. Run W01/W02 download first.")

def sql_path(p: Path) -> str:
    return "'" + p.resolve().as_posix().replace("'", "''") + "'"

con = duckdb.connect(str(DB_PATH))

con.execute("DROP VIEW IF EXISTS raw_ps")
con.execute(f"CREATE VIEW raw_ps AS SELECT * FROM read_csv_auto({sql_path(RAW_CSV)})")

con.sql("SELECT COUNT(*) AS n_raw FROM raw_ps").show()

┌───────┐
│ n_raw │
│ int64 │
├───────┤
│  6291 │
└───────┘



## Parte A — Limpieza Raw→Silver v2

In [2]:
con.sql("SELECT discoverymethod, COUNT(*) AS n FROM raw_ps WHERE discoverymethod IS NOT NULL GROUP BY discoverymethod ORDER BY n DESC LIMIT 15").show()
con.sql("SELECT COUNT(DISTINCT discoverymethod) AS n_unique_methods FROM raw_ps WHERE discoverymethod IS NOT NULL").show()

┌───────────────────────────────┬───────┐
│        discoverymethod        │   n   │
│            varchar            │ int64 │
├───────────────────────────────┼───────┤
│ Transit                       │  4651 │
│ Radial Velocity               │  1181 │
│ Microlensing                  │   278 │
│ Imaging                       │    97 │
│ Transit Timing Variations     │    41 │
│ Eclipse Timing Variations     │    17 │
│ Orbital Brightness Modulation │     9 │
│ Pulsar Timing                 │     8 │
│ Astrometry                    │     6 │
│ Pulsation Timing Variations   │     2 │
│ Disk Kinematics               │     1 │
├───────────────────────────────┴───────┤
│ 11 rows                     2 columns │
└───────────────────────────────────────┘

┌──────────────────┐
│ n_unique_methods │
│      int64       │
├──────────────────┤
│               11 │
└──────────────────┘



### TODO A1 — method_map

In [3]:
con.execute("DROP TABLE IF EXISTS method_map")

con.execute("""
CREATE TABLE method_map (
  raw_method VARCHAR PRIMARY KEY,
  canonical_method VARCHAR NOT NULL
)
""")

con.execute("""
INSERT INTO method_map VALUES
  ('Transit', 'transit'),
  ('Radial Velocity', 'radial_velocity'),
  ('Microlensing', 'microlensing'),
  ('Imaging', 'imaging'),
  ('Transit Timing Variations', 'transit_timing_variations'),
  ('Eclipse Timing Variations', 'eclipse_timing_variations'),
  ('Orbital Brightness Modulation', 'orbital_brightness_modulation'),
  ('Pulsar Timing', 'pulsar_timing')
""")

con.sql("SELECT * FROM method_map ORDER BY raw_method").show()

┌───────────────────────────────┬───────────────────────────────┐
│          raw_method           │       canonical_method        │
│            varchar            │            varchar            │
├───────────────────────────────┼───────────────────────────────┤
│ Eclipse Timing Variations     │ eclipse_timing_variations     │
│ Imaging                       │ imaging                       │
│ Microlensing                  │ microlensing                  │
│ Orbital Brightness Modulation │ orbital_brightness_modulation │
│ Pulsar Timing                 │ pulsar_timing                 │
│ Radial Velocity               │ radial_velocity               │
│ Transit                       │ transit                       │
│ Transit Timing Variations     │ transit_timing_variations     │
└───────────────────────────────┴───────────────────────────────┘



### TODO A2 — silver_planet_v2

In [4]:
con.execute("DROP TABLE IF EXISTS silver_planet_v2")

con.execute("""
CREATE TABLE silver_planet_v2 AS
WITH cleaned AS (
  SELECT
    pl_name,
    hostname,
    LOWER(TRIM(hostname)) AS hostname_clean,
    discoverymethod,
    TRIM(discoverymethod) AS discoverymethod_norm,
    disc_year,
    pl_orbper,
    pl_rade,
    pl_bmasse,
    pl_eqt,
    sy_dist,
    ra,
    dec
  FROM raw_ps
  WHERE pl_name IS NOT NULL
    AND hostname IS NOT NULL
)
SELECT
  c.pl_name,
  c.hostname,
  c.hostname_clean,
  c.discoverymethod,
  COALESCE(m.canonical_method, LOWER(TRIM(c.discoverymethod_norm))) AS discoverymethod_clean,
  c.disc_year,
  CASE
    WHEN c.disc_year IS NULL THEN 'unknown'
    WHEN c.disc_year < 2000 THEN 'pre_2000'
    WHEN c.disc_year BETWEEN 2000 AND 2009 THEN '2000s'
    WHEN c.disc_year BETWEEN 2010 AND 2019 THEN '2010s'
    WHEN c.disc_year >= 2020 THEN '2020s'
    ELSE 'unknown'
  END AS disc_era,
  c.pl_orbper,
  c.pl_rade,
  c.pl_bmasse,
  c.pl_eqt,
  c.sy_dist,
  c.ra,
  c.dec
FROM cleaned c
LEFT JOIN method_map m
  ON c.discoverymethod_norm = m.raw_method
WHERE (c.disc_year IS NULL OR c.disc_year BETWEEN 1980 AND 2026)
  AND (c.pl_rade IS NULL OR c.pl_rade > 0)
  AND (c.pl_bmasse IS NULL OR c.pl_bmasse > 0)
""")

con.sql("SELECT COUNT(*) AS n_rows FROM silver_planet_v2").show()

con.sql("""
SELECT COUNT(*) AS n_null_hosts
FROM silver_planet_v2
WHERE hostname_clean IS NULL
""").show()

con.sql("""
SELECT discoverymethod_clean, COUNT(*) AS n
FROM silver_planet_v2
WHERE discoverymethod_clean IS NOT NULL
GROUP BY discoverymethod_clean
ORDER BY n DESC
LIMIT 15
""").show()

┌────────┐
│ n_rows │
│ int64  │
├────────┤
│   6291 │
└────────┘

┌──────────────┐
│ n_null_hosts │
│    int64     │
├──────────────┤
│            0 │
└──────────────┘

┌───────────────────────────────┬───────┐
│     discoverymethod_clean     │   n   │
│            varchar            │ int64 │
├───────────────────────────────┼───────┤
│ transit                       │  4651 │
│ radial_velocity               │  1181 │
│ microlensing                  │   278 │
│ imaging                       │    97 │
│ transit_timing_variations     │    41 │
│ eclipse_timing_variations     │    17 │
│ orbital_brightness_modulation │     9 │
│ pulsar_timing                 │     8 │
│ astrometry                    │     6 │
│ pulsation timing variations   │     2 │
│ disk kinematics               │     1 │
├───────────────────────────────┴───────┤
│ 11 rows                     2 columns │
└───────────────────────────────────────┘



## Parte B — Many-to-Many (toy schema)

In [5]:
con.execute("DROP TABLE IF EXISTS planet_method_demo")
con.execute("DROP TABLE IF EXISTS method_demo")
con.execute("DROP TABLE IF EXISTS planet_demo")

con.execute("""
CREATE TABLE planet_demo (
  planet_id INTEGER PRIMARY KEY,
  name VARCHAR NOT NULL
)
""")

con.execute("""
CREATE TABLE method_demo (
  method_id INTEGER PRIMARY KEY,
  method_name VARCHAR NOT NULL UNIQUE
)
""")

con.execute("""
CREATE TABLE planet_method_demo (
  planet_id INTEGER NOT NULL,
  method_id INTEGER NOT NULL,
  PRIMARY KEY (planet_id, method_id),
  FOREIGN KEY (planet_id) REFERENCES planet_demo(planet_id),
  FOREIGN KEY (method_id) REFERENCES method_demo(method_id)
)
""")

con.execute("""
INSERT INTO planet_demo VALUES
  (1, 'Kepler-22 b'),
  (2, 'TRAPPIST-1 e'),
  (3, '51 Pegasi b'),
  (4, 'HR 8799 b')
""")

con.execute("""
INSERT INTO method_demo VALUES
  (10, 'transit'),
  (20, 'radial_velocity'),
  (30, 'imaging')
""")

con.execute("""
INSERT INTO planet_method_demo VALUES
  (1, 10),
  (1, 20),
  (2, 10),
  (3, 20),
  (4, 30),
  (4, 20)
""")

In [7]:
q1 = """
SELECT
  m.method_name,
  COUNT(DISTINCT pm.planet_id) AS n_planets
FROM method_demo m
JOIN planet_method_demo pm
  ON m.method_id = pm.method_id
GROUP BY m.method_name
ORDER BY n_planets DESC, m.method_name
"""

con.sql(q1).show()

┌─────────────────┬───────────┐
│   method_name   │ n_planets │
│     varchar     │   int64   │
├─────────────────┼───────────┤
│ radial_velocity │         3 │
│ transit         │         2 │
│ imaging         │         1 │
└─────────────────┴───────────┘



In [8]:
q2 = """
SELECT
  p.name AS planet_name,
  COUNT(DISTINCT pm.method_id) AS n_methods
FROM planet_demo p
JOIN planet_method_demo pm
  ON p.planet_id = pm.planet_id
GROUP BY p.name
ORDER BY n_methods DESC, planet_name
"""

con.sql(q2).show()

┌──────────────┬───────────┐
│ planet_name  │ n_methods │
│   varchar    │   int64   │
├──────────────┼───────────┤
│ HR 8799 b    │         2 │
│ Kepler-22 b  │         2 │
│ 51 Pegasi b  │         1 │
│ TRAPPIST-1 e │         1 │
└──────────────┴───────────┘



In [ ]:
# TODO B2 (REQUERIDO): check de duplicados en la link table (debe dar 0 filas)
# Si tu PK compuesta está bien, este check debería retornar vacío.

con.sql("""
SELECT planet_id, method_id, COUNT(*) AS c
FROM planet_method_demo
GROUP BY planet_id, method_id
HAVING COUNT(*) > 1
""").show()

# (Opcional) intenta insertar un duplicado para ver que la PK compuesta lo bloquea
# try:
#     con.execute("INSERT INTO planet_method_demo VALUES (1, 10)")
# except Exception as e:
#     print("OK (PK compuesta bloquea duplicado):", str(e).splitlines()[0])

┌───────────┬───────────┬───────┐
│ planet_id │ method_id │   c   │
│   int32   │   int32   │ int64 │
├───────────┴───────────┴───────┤
│            0 rows             │
└───────────────────────────────┘



## Entregable único semanal (W08)
- Ejecuta el assignment.
- Entrega:
  1) `docs/w08_report.md` (copiar template)
  2) 1 entrada nueva en `docs/decisions_log.md` (copiar template)

**Extra requerido (M:N):** incluye evidencia de PK/FK en tu DDL y/o el check `HAVING COUNT(*)>1` retornando vacío.